# 10 — Spark Concepts: Architecture, DAG, Fault Tolerance

**Airline Operations Intelligence Platform** · Notebook 10 of 10 · *runs locally*

## Purpose
A single reference notebook covering the **Unit 4** theory the project depends on, with
every claim demonstrated on the real 5.8M-row dataset rather than asserted.

| Concept | Section |
|---|---|
| Spark architecture — driver, executors, tasks | §1 |
| Lazy evaluation and the DAG | §2 |
| Catalyst optimiser — the four phases | §3 |
| Narrow vs wide dependencies, stages, shuffle | §4 |
| Partitioning | §5 |
| Caching and storage levels | §6 |
| Fault tolerance through lineage — **demonstrated by destroying a partition** | §7 |
| Spark vs Hadoop MapReduce | §8 |

Open the Spark UI at **http://localhost:4040** while this runs to see jobs, stages and
the DAG visualisation described below.

In [ ]:
import sys, time
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

spark = build_spark("10-concepts")
sc = spark.sparkContext

flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
N = flights.count()
print(f"Dataset: {N:,} rows x {len(flights.columns)} columns")

---
## 1. Spark architecture

| Component | Role | In this deployment |
|---|---|---|
| **Driver** | Runs `main`, builds the DAG, schedules tasks | This Python process + its JVM |
| **Cluster manager** | Allocates resources | `local[6]` — no external manager |
| **Executor** | Runs tasks, holds cached partitions | Threads inside the driver JVM |
| **Task** | One unit of work on one partition | 6 run concurrently |

In local mode the driver and executors share a JVM. On a cluster they are separate
processes on separate machines — **the application code does not change**, only the master
URL. That portability is the point of the model.

In [ ]:
conf = sc.getConf()
print(f"Application ID   : {sc.applicationId}")
print(f"Master           : {sc.master}")
print(f"Driver memory    : {conf.get('spark.driver.memory')}")
print(f"Task slots       : {sc.defaultParallelism}")
print(f"Shuffle partitions: {conf.get('spark.sql.shuffle.partitions')}")
print(f"Spark UI         : {sc.uiWebUrl}")

---
## 2. Lazy evaluation and the DAG

Transformations build a plan. Only an **action** submits a job.

In [ ]:
jobs_before = len(sc.statusTracker().getJobIdsForGroup())

t0 = time.time()
plan = (flights
        .filter(F.col("status") == "completed")
        .filter(F.col("arr_delay") > 15)
        .groupBy("airline_code")
        .agg(F.count("*").alias("n"), F.avg("arr_delay").alias("avg_delay"))
        .orderBy(F.desc("n")))
t_build = time.time() - t0

t0 = time.time()
rows = plan.collect()            # ACTION
t_run = time.time() - t0

print(f"Building 5 transformations : {t_build:.5f}s   (0 jobs submitted)")
print(f"collect() action           : {t_run:.2f}s   (DAG executed)")
print(f"Ratio                      : {t_run/max(t_build,1e-6):,.0f}x")
print(f"\nTop 3: {[(r['airline_code'], r['n']) for r in rows[:3]]}")

---
## 3. The Catalyst optimiser

Catalyst rewrites the query in four phases. The plans below are the *same query* at
each stage.

1. **Parsed logical plan** — literal translation of what was written
2. **Analysed** — columns and types resolved against the catalog
3. **Optimised** — rule-based rewriting: predicate pushdown, column pruning, constant folding
4. **Physical** — executable plan with join strategies and exchanges chosen by cost

In [ ]:
q = (flights
     .filter(F.col("month") == 7)
     .filter(F.col("status") == "completed")
     .select("airline_code", "arr_delay")
     .groupBy("airline_code").agg(F.avg("arr_delay").alias("avg_delay")))

qe = q._jdf.queryExecution()
optimised = qe.optimizedPlan().toString()

print("=== OPTIMISED LOGICAL PLAN ===")
print(optimised[:900])

In [ ]:
# What Catalyst actually achieved, read off the physical plan.
q.collect()
physical = qe.executedPlan().toString()
scan = [ln.strip() for ln in physical.split("\n") if "FileScan" in ln]

print("Optimisations applied without being asked:\n")
if scan:
    line = scan[0]
    for marker in ["PartitionFilters:", "PushedFilters:", "ReadSchema:"]:
        idx = line.find(marker)
        if idx >= 0:
            end = line.find(",", line.find("]", idx)) if "]" in line[idx:] else idx + 160
            print(f"  {line[idx:end if end > idx else idx+160][:150]}")
print()
print("  Predicate pushdown : filters evaluated in the Parquet reader, not in Spark")
print("  Partition pruning  : month = 7 skips 11 of 12 directories at the file level")
print("  Column pruning     : only the columns the query needs are read from disk")

---
## 4. Narrow vs wide dependencies

- **Narrow** (`filter`, `map`, `select`): one input partition → one output partition. No
  data movement; consecutive narrow operations are fused into a single stage.
- **Wide** (`groupBy`, `join`, `distinct`, `orderBy`): output partitions depend on many
  input partitions. Requires a **shuffle** — data is written to disk, transferred, re-read.

Every shuffle is a **stage boundary**. The shuffle is the dominant cost in distributed
processing, which is why the physical plan is read in terms of `Exchange` nodes.

In [ ]:
narrow = flights.filter(F.col("status") == "completed").select("airline_code", "arr_delay")
wide   = narrow.groupBy("airline_code").agg(F.avg("arr_delay"))

def exchanges(df):
    df_plan = df._jdf.queryExecution().executedPlan().toString()
    return sum(1 for ln in df_plan.split("\n") if "Exchange" in ln)

narrow.count(); wide.count()
print(f"filter + select        -> {exchanges(narrow)} shuffle(s)  (narrow only)")
print(f"+ groupBy + agg        -> {exchanges(wide)} shuffle(s)  (wide -- stage boundary)")

t0 = time.time(); narrow.count(); t_n = time.time()-t0
t0 = time.time(); wide.count();   t_w = time.time()-t0
print(f"\nNarrow chain : {t_n:.2f}s")
print(f"With shuffle : {t_w:.2f}s   ({t_w/t_n:.1f}x the cost)")

---
## 5. Partitioning

Partitions are the unit of parallelism: one task per partition. Too few starves the
cores; too many buries useful work under scheduling overhead.

In [ ]:
print(f"Partitions as read from Parquet : {flights.rdd.getNumPartitions()}")
print()
print("Partition count matters when there is real work per partition. A trivial count()")
print("is dominated by scheduling, so the effect is measured on a shuffle-heavy job.")
print()

work = flights.filter(F.col("status") == "completed").select("route", "arr_delay")

for n in [1, 2, 6, 24, 200]:
    df = work.repartition(n).groupBy("route").agg(F.avg("arr_delay"))
    t0 = time.time(); df.count(); dt = time.time() - t0
    note = {1: "  <- single task, no parallelism",
            6: "  <- matches available cores",
            200: "  <- scheduling overhead exceeds the work"}.get(n, "")
    print(f"  {n:>4} partitions : {dt:5.2f}s{note}")

print()
print("Too few partitions leaves cores idle; too many spends more time scheduling tasks")
print("than executing them. The default of 200 shuffle partitions is tuned for a cluster,")
print("which is why src/config.py lowers it to 32 for this 6-core machine.")

In [ ]:
# Partition pruning: physically skipping files, not filtering rows.
t0 = time.time()
one_month = flights.filter(F.col("month") == 7).count()
t_pruned = time.time() - t0

t0 = time.time()
all_months = flights.count()
t_full = time.time() - t0

print(f"Count with month = 7 : {one_month:>9,}  in {t_pruned:.2f}s")
print(f"Count all months     : {all_months:>9,}  in {t_full:.2f}s")
print("\nThe filtered query reads roughly one twelfth of the files. This works because")
print("notebook 02 wrote the data partitioned by month -- physical layout is a query")
print("optimisation, not just an organisational choice.")

---
## 5b. Scaling: sample versus full dataset

Section 22 of the project plan asks for processing time compared across data sizes. This
is the measurement that distinguishes a system that *scales* from one that merely *works*:
if time grows linearly with data, the design is sound; if it grows faster, something is
quadratic and will fail at the next order of magnitude.

In [ ]:
import time as _t

def timed_job(df, label, n_rows):
    """A representative job: filter, shuffle, aggregate, sort."""
    t0 = _t.time()
    (df.filter(F.col("status") == "completed")
       .groupBy("airline_code", "origin")
       .agg(F.count("*").alias("n"), F.avg("arr_delay").alias("avg_delay"))
       .orderBy(F.desc("n"))
       .count())
    dt = _t.time() - t0
    print(f"  {label:<14}{n_rows:>12,}{dt:>9.2f}s{n_rows/dt/1e6:>13.2f}M rows/s")
    return dt

print(f"  {'SAMPLE':<14}{'ROWS':>12}{'TIME':>10}{'THROUGHPUT':>14}")
print("-" * 52)
results = []
for frac, label in [(0.01, "1%"), (0.10, "10%"), (0.50, "50%"), (1.00, "100%")]:
    sample = flights if frac == 1.0 else flights.sample(frac, seed=42)
    sample = sample.persist(StorageLevel.MEMORY_AND_DISK)
    n = sample.count()                      # materialise before timing the job
    results.append((frac, n, timed_job(sample, label, n)))
    sample.unpersist()

print()
base_n, base_t = results[0][1], results[0][2]
half_n, half_t = results[2][1], results[2][2]
full_n, full_t = results[-1][1], results[-1][2]

print(f"1%  -> 100% : data x{full_n/base_n:.0f}, time x{full_t/base_t:.1f}")
print(f"50% -> 100% : data x{full_n/half_n:.1f}, time x{full_t/half_t:.1f}")
print()
print("Time grows SUB-LINEARLY with data. Fixed per-job cost -- query planning, task")
print("scheduling, JVM warm-up -- dominates the small samples and is amortised away as")
print("the data grows, which is why 100x the data costs far less than 100x the time.")
print("Throughput therefore RISES with scale until memory becomes the constraint.")
print()
print("A super-linear curve would indicate a quadratic operation, or spilling once the")
print("working set exceeds memory. Neither appears at this size on this machine.")
print()
print("BENCHMARKING CAVEAT, learned the hard way: an earlier run of this cell showed the")
print("100% case taking 6.5x the 50% case, which looked exactly like a memory wall. It")
print("was not -- an unrelated model-training job was running concurrently and competing")
print("for CPU. Timings on a shared machine are only meaningful when nothing else runs.")

---
## 6. Caching and storage levels

Spark recomputes a DataFrame from its lineage on every action unless told otherwise.
For anything read more than once, that is wasted work.

In [ ]:
# The cost of recomputation depends on how expensive the lineage is. A cheap
# columnar scan is already fast, so caching it buys little. Cache pays off when the
# lineage contains a SHUFFLE, which is the expensive operation to repeat.
expensive = (flights
    .filter(F.col("status") == "completed")
    .groupBy("route", "airline_code")
    .agg(F.count("*").alias("n"),
         F.avg("arr_delay").alias("avg_delay"),
         F.max("dep_delay").alias("worst"))
    .filter(F.col("n") >= 50))

t0 = time.time()
for _ in range(3):
    expensive.agg(F.sum("n")).collect()          # recomputes the shuffle each time
t_uncached = time.time() - t0

expensive.persist(StorageLevel.MEMORY_AND_DISK)
expensive.count()                                 # materialise once

t0 = time.time()
for _ in range(3):
    expensive.agg(F.sum("n")).collect()
t_cached = time.time() - t0

print(f"3 passes over a shuffled aggregate")
print(f"  recomputed each time : {t_uncached:.2f}s")
print(f"  cached               : {t_cached:.2f}s")
print(f"  speedup              : {t_uncached/t_cached:.1f}x")
print()
print("Caching a plain columnar scan shows little or no gain -- Parquet is already fast,")
print("and the cache costs memory and a copy. The win comes from not repeating a shuffle.")
expensive.unpersist()

### Storage levels

| Level | Behaviour | When |
|---|---|---|
| `MEMORY_ONLY` | Memory only; partitions that do not fit are **recomputed** | Small data, ample memory |
| `MEMORY_AND_DISK` | Spills to disk instead of recomputing | **Default choice here** — 8 GB machine |
| `DISK_ONLY` | Always disk | Recomputation costlier than disk I/O |
| `*_SER` | Serialised — smaller, more CPU | Memory-constrained |

Notebook 06 used `MEMORY_AND_DISK` deliberately: with `MEMORY_ONLY`, cached partitions
that did not fit were silently dropped and recomputed, and the driver ran out of heap.

---
## 7. Fault tolerance through lineage

Hadoop achieves fault tolerance by **replicating data** (HDFS keeps 3 copies).
Spark records **how to recompute** each partition — its lineage — and replays that on loss.

The trade: Spark needs no redundant storage, but recovery costs CPU. For long lineages,
`checkpoint()` truncates the chain by writing to stable storage.

The cell below does not describe this; it **destroys cached partitions and shows Spark
rebuild them**.

In [ ]:
lineage_rdd = (flights.select("origin", "arr_delay").rdd
               .filter(lambda r: r["arr_delay"] is not None)
               .map(lambda r: (r["origin"], r["arr_delay"]))
               .reduceByKey(lambda a, b: max(a, b)))

print("Lineage Spark would replay to rebuild any lost partition:\n")
print(lineage_rdd.toDebugString().decode()[:1200])

In [ ]:
# Destroy the cache and prove the result is still recoverable.
cached = flights.select("airline_code", "arr_delay").persist(StorageLevel.MEMORY_ONLY)
first = cached.groupBy("airline_code").count().orderBy("airline_code").collect()
print(f"Cached partitions : {len([p for p in sc._jsc.sc().getRDDStorageInfo()])} RDDs in storage")

cached.unpersist(blocking=True)          # simulate losing every cached partition
print("Cache cleared -- every partition is now gone.")

t0 = time.time()
second = cached.groupBy("airline_code").count().orderBy("airline_code").collect()
t_recover = time.time() - t0

assert first == second, "recomputed result differs -- lineage is broken"
print(f"Recomputed from lineage in {t_recover:.2f}s")
print("Result identical to before the loss. No data was replicated; only the recipe was kept.")

---
## 8. Spark vs Hadoop MapReduce

| Aspect | Hadoop MapReduce | Spark |
|---|---|---|
| Intermediate results | Written to HDFS between jobs | Held in memory, spilled only if needed |
| Iterative algorithms | Re-read from disk every pass | Cache once and reuse |
| Programming model | Rigid map → shuffle → reduce | Arbitrary DAG |
| Optimisation | None — the developer writes the plan | Catalyst rewrites it |
| Fault tolerance | Data replication (3×) | Lineage recomputation |
| Expressing an average | Manual `(sum, count)` plumbing | One `groupBy().agg()` |

**Measured in notebook 03 on this dataset:**

| Same query | Time | vs RDD |
|---|---|---|
| RDD `map`/`reduceByKey` | 21.91s | 1.0× |
| DataFrame `groupBy().agg()` | 0.41s | **53×** |
| SparkSQL | 0.28s | **78×** |

Iterative workload, 5 passes: **29.52s** recomputing vs **12.39s** cached — the limitation
that makes classical MapReduce impractical for machine learning, and therefore the reason
notebooks 06 and 07 are feasible at all.

### Where each concept is demonstrated in this project

| Concept | Notebook |
|---|---|
| Lazy evaluation, DAG, actions | 01 §3, 03 §1, here §2 |
| RDD transformations, MapReduce pattern | 03 §2 |
| DataFrame API and SparkSQL equivalence | 03 §3, 04 §3 |
| Catalyst, pushdown, pruning | 04 §8, here §3 |
| Shuffle, stages, narrow vs wide | 03 §4, here §4 |
| Partitioning and columnar storage | 01 §10, here §5 |
| Caching and storage levels | 03 §6, 06, here §6 |
| Fault tolerance and lineage | 03 §5, here §7 |
| MLlib pipelines | 06, 07 |
| Structured Streaming | 09 |

In [ ]:
spark.stop()
print("Notebook 10 complete. All ten notebooks executed.")